# 4장 2강: Transformer 아키텍처 — 실습문제 풀이

> **풀이 기준**: 업로드된 `jena_climate_2009_2016.csv`의 처음 6행으로 코드를 실제 실행한 결과입니다. 출력은 소수 여섯째 자리에서 반올림했습니다.
>
> **가정한 내용**: 별도 판단이 필요한 부분은 없었습니다. 제공 함수·가중치를 그대로 사용했고, 문제 2-2의 Decoder 구조는 문제에 명시된 원래 Encoder–Decoder Transformer(하위 층 뒤 정규화)를 기준으로 설명했습니다.

---

## 실습 준비 (제공 코드)

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("jena_climate_2009_2016.csv", nrows=6)
times = pd.to_datetime(df["Date Time"], format="%d.%m.%Y %H:%M:%S")
raw = df[["T (degC)", "rh (%)"]].to_numpy(dtype=float)
X_all = raw / np.array([10.0, 100.0])
X = X_all[:3]
X_home = X_all[3:6]

print("읽은 데이터 크기:", df.shape)
print(df[["Date Time", "T (degC)", "rh (%)"]].to_string(index=False))
print("선택 구간 간격(분):", times.diff().dropna().dt.total_seconds().unique() / 60)
print("선택 두 열의 결측값 수:", df[["T (degC)", "rh (%)"]].isna().sum().sum())
print("실습 X:", X)
print("과제 X_home:", X_home)

읽은 데이터 크기: (6, 15)
          Date Time  T (degC)  rh (%)
01.01.2009 00:10:00     -8.02    93.3
01.01.2009 00:20:00     -8.41    93.4
01.01.2009 00:30:00     -8.51    93.9
01.01.2009 00:40:00     -8.31    94.2
01.01.2009 00:50:00     -8.27    94.1
01.01.2009 01:00:00     -8.05    94.4
선택 구간 간격(분): [10.]
선택 두 열의 결측값 수: 0
실습 X: [[-0.802  0.933]
 [-0.841  0.934]
 [-0.851  0.939]]
과제 X_home: [[-0.831  0.942]
 [-0.827  0.941]
 [-0.805  0.944]]


In [2]:
def position_encoding(length):
    positions = np.arange(length, dtype=float)
    return np.column_stack([np.sin(positions), np.cos(positions)])


def softmax_rows(scores):
    shifted = scores - np.max(scores, axis=1, keepdims=True)
    exp_scores = np.exp(shifted)
    return exp_scores / np.sum(exp_scores, axis=1, keepdims=True)


def attention_head(values, wq, wk, wv):
    q = values @ wq
    k = values @ wk
    v = values @ wv
    d_k = k.shape[1]
    scores = q @ k.T / np.sqrt(d_k)
    weights = softmax_rows(scores)
    head_output = weights @ v
    return weights, head_output


print("위치 정보와 Attention 함수 준비 완료")

위치 정보와 Attention 함수 준비 완료


In [3]:
WQ1 = np.array([[1.0], [0.0]])
WK1 = np.array([[1.0], [0.0]])
WV1 = np.array([[1.0], [0.0]])
WQ2 = np.array([[0.0], [1.0]])
WK2 = np.array([[0.0], [1.0]])
WV2 = np.array([[0.0], [1.0]])
WO = np.array([[1.0, 0.5], [0.5, 1.0]])

print("각 헤드 가중치 크기:", WQ1.shape, WQ2.shape)
print("출력 가중치 WO:", WO)

각 헤드 가중치 크기: (2, 1) (2, 1)
출력 가중치 WO: [[1.  0.5]
 [0.5 1. ]]


---

## 실습 1. Positional Encoding

### 문제 1-1. 관측값에 위치 정보 더하기

In [4]:
PE = position_encoding(len(X))
X_pos = X + PE

print("X:\n", X, "\nshape:", X.shape)
print("PE:\n", np.round(PE, 6), "\nshape:", PE.shape)
print("X_pos:\n", np.round(X_pos, 6), "\nshape:", X_pos.shape)

X:
 [[-0.802  0.933]
 [-0.841  0.934]
 [-0.851  0.939]] 
shape: (3, 2)
PE:
 [[ 0.        1.      ]
 [ 0.841471  0.540302]
 [ 0.909297 -0.416147]] 
shape: (3, 2)
X_pos:
 [[-8.020000e-01  1.933000e+00]
 [ 4.710000e-04  1.474302e+00]
 [ 5.829700e-02  5.228530e-01]] 
shape: (3, 2)


#### 답변

**계산·결과 정리**

| 위치 p | X | PE = [sin p, cos p] | X_pos = X + PE |
|---:|---|---|---|
| 0 | [−0.802, 0.933] | [0, 1] | [−0.802, 1.933] |
| 1 | [−0.841, 0.934] | [0.841471, 0.540302] | [0.000471, 1.474302] |
| 2 | [−0.851, 0.939] | [0.909297, −0.416147] | [0.058297, 0.522853] |

- 첫째 행: [−0.802 + 0, 0.933 + 1] = **[−0.802, 1.933]**
- 둘째 행: [−0.841 + 0.841471, 0.934 + 0.540302] = **[0.000471, 1.474302]**
- X, PE, X_pos 모두 크기는 **(3, 2)**입니다.

**Q1. Positional Encoding은 왜 필요한가요? PE를 더하면 입력 열 수가 늘어나나요?**

- Self-Attention은 모든 위치를 한꺼번에 비교하는 계산이라, 입력 순서를 바꿔도 각 행의 계산 결과가 그대로 자리만 바뀝니다. 즉 스스로는 "몇 번째 위치인지"를 알지 못하므로, 위치 정보를 입력 표현에 넣어 줘야 합니다.
- 늘어나지 않습니다. PE는 X와 같은 (3, 2) 크기로 만들어 **같은 위치끼리 더하므로** 열 수는 2 그대로입니다.

**Q2. 위치 0·1·2의 PE는 각각 무엇인가요?**

- 위치 0: [sin 0, cos 0] = **[0, 1]**
- 위치 1: [sin 1, cos 1] ≈ **[0.841471, 0.540302]**
- 위치 2: [sin 2, cos 2] ≈ **[0.909297, −0.416147]**
- X_pos 첫째·둘째 행의 대입은 위 표와 같습니다.

**Q3. 위치 1은 1분 또는 1시라는 뜻인가요? 첫째 행의 두 번째 값 1.933을 상대습도 193.3%라고 읽어도 되나요?**

- 아닙니다. p는 **구간 안의 순번**(0번째, 1번째, 2번째)일 뿐 실제 시각(00:20)이나 시간 단위가 아닙니다.
- 안 됩니다. 1.933은 습도/100(0.933)에 위치 표현 1을 더한 **내부 특징 값**입니다. 위치 정보가 섞였으므로 더 이상 원래 단위의 습도로 해석할 수 없습니다.

---

## 실습 2. Multi-Head Attention과 Encoder·Decoder

### 문제 2-1. 두 헤드의 결과를 결합하기

In [5]:
A1, H1 = attention_head(X_pos, WQ1, WK1, WV1)
A2, H2 = attention_head(X_pos, WQ2, WK2, WV2)

C = np.concatenate([H1, H2], axis=1)
M = C @ WO

print("A1:\n", np.round(A1, 6), " shape:", A1.shape, " 행 합:", A1.sum(axis=1))
print("H1:\n", np.round(H1, 6), " shape:", H1.shape)
print("A2:\n", np.round(A2, 6), " shape:", A2.shape, " 행 합:", A2.sum(axis=1))
print("H2:\n", np.round(H2, 6), " shape:", H2.shape)
print("C:\n", np.round(C, 6), " shape:", C.shape)
print("M:\n", np.round(M, 6), " shape:", M.shape)

A1:
 [[0.493339 0.259204 0.247457]
 [0.333246 0.333372 0.333381]
 [0.322651 0.338104 0.339245]]  shape: (3, 3)  행 합: [1. 1. 1.]
H1:
 [[-0.38111 ]
 [-0.247671]
 [-0.23883 ]]  shape: (3, 1)
A2:
 [[0.676809 0.278865 0.044326]
 [0.612155 0.31129  0.076554]
 [0.441469 0.34733  0.211201]]  shape: (3, 3)  행 합: [1. 1. 1.]
H2:
 [[1.742579]
 [1.682259]
 [1.475857]]  shape: (3, 1)
C:
 [[-0.38111   1.742579]
 [-0.247671  1.682259]
 [-0.23883   1.475857]]  shape: (3, 2)
M:
 [[0.49018  1.552024]
 [0.593458 1.558423]
 [0.499099 1.356442]]  shape: (3, 2)


#### 답변

**첫 행 계산 과정**

- H1[0, 0] ≈ **−0.381110**, H2[0, 0] ≈ **1.742579**
- C 첫 행 = [−0.381110, 1.742579]
- M 첫 행 = [−0.381110 × 1 + 1.742579 × 0.5, −0.381110 × 0.5 + 1.742579 × 1]
  = [−0.381110 + 0.871289, −0.190555 + 1.742579] = **[0.490180, 1.552024]**

**Q1. 두 헤드가 같은 입력을 읽는데 Attention 가중치가 다른 이유는 무엇인가요? 각 헤드의 Q·K·V와 Attention 가중치·헤드 출력의 크기는 무엇인가요?**

- 각 헤드가 **서로 다른 가중치**로 Q·K·V를 만들기 때문입니다. 헤드 1은 X_pos의 첫 성분, 헤드 2는 둘째 성분을 선택하므로 비교하는 값이 달라져 점수와 비중도 달라집니다.
- 각 헤드의 Q·K·V: **(3, 1)**, Attention 가중치 A1·A2: **(3, 3)**, 헤드 출력 H1·H2: **(3, 1)**

**Q2. 각 헤드의 d_k는 얼마이며 점수를 무엇으로 나누나요? A1·A2의 각 행 합은 얼마인가요?**

- 각 헤드의 K는 열이 1개이므로 d_k = **1**이고, 점수를 **√1 = 1**로 나눕니다. d_model = 2나 헤드 수 2가 기준이 아닙니다.
- A1·A2의 각 행 합은 모두 **1**입니다.

**Q3. 첫 위치의 H1·H2, 연결 결과 C, 출력 M은 각각 무엇인가요?**

- H1 ≈ −0.381110, H2 ≈ 1.742579, C = [−0.381110, 1.742579], M ≈ **[0.490180, 1.552024]**입니다(위 계산 과정).

**Q4. Multi-Head Attention은 같은 계산을 단순 반복해 평균내는 것인가요? 두 헤드를 차례로 쌓은 층 두 개와 같나요?**

- 아닙니다. 각 헤드는 **서로 다른 가중치**로 다른 관점의 Attention을 계산하고, 결과를 평균하지 않고 **옆으로 연결(Concat)한 뒤 WO로 다시 섞습니다**.
- 층 두 개와도 다릅니다. 두 헤드는 **같은 입력 X_pos에서 나란히(병렬로)** 출발하며, 헤드 2가 헤드 1의 출력을 입력으로 받지 않습니다.

---

### 문제 2-2. Encoder와 Decoder의 정보 흐름 연결하기 (코드 없음)

#### 답변

**처리 순서**

- **Encoder**: 입력 표현(00:10·00:20·00:30) + PE → Multi-Head Self-Attention → Add & Norm → FFN → Add & Norm → Encoder 출력
- **Decoder**: 이동한 입력(−8.51, −8.31, −8.27) 표현 + PE → Masked Multi-Head Self-Attention → Add & Norm → Encoder–Decoder Attention → Add & Norm → FFN → Add & Norm → 최종 출력층 → 00:40·00:50·01:00 기온

**Q·K·V 출처 표**

| Attention | Query | Key | Value |
|---|---|---|---|
| Encoder Self-Attention | Encoder 입력(이전 층) | Encoder 입력 | Encoder 입력 |
| Decoder Masked Self-Attention | Decoder 입력(이전 층) | Decoder 입력 (현재 위치까지만) | Decoder 입력 (현재 위치까지만) |
| Encoder–Decoder Attention (Cross) | **Decoder** (직전 단계 결과) | **Encoder 출력** | **Encoder 출력** |

**Decoder 위치별 참고 허용 표 (✓ 허용, ✗ 차단)**

| Query 위치 \ 입력 위치 | 0 (−8.51) | 1 (−8.31) | 2 (−8.27) |
|---|:---:|:---:|:---:|
| 0 (00:40 예측) | ✓ | ✗ | ✗ |
| 1 (00:50 예측) | ✓ | ✓ | ✗ |
| 2 (01:00 예측) | ✓ | ✓ | ✓ |

**Q1. Encoder와 Decoder의 처리 순서는 각각 무엇인가요? FFN과 Add & Norm은 어떤 역할인가요?**

- 처리 순서는 위와 같습니다.
- FFN: 각 위치의 표현을 **같은 가중치의 작은 신경망**으로 위치마다 따로 변환해, Attention으로 모은 정보를 더 가공합니다.
- Add & Norm: 하위 층의 입력을 결과에 **더하고(잔차 연결)**, 위치별 특징값을 **정규화**해 정보 손실을 줄이고 학습을 안정시킵니다.

**Q2. 세 Attention의 Q·K·V 출처는 각각 어디인가요? Decoder가 Encoder의 정보를 읽는 단계는 무엇인가요?**

- 출처는 위 표와 같습니다.
- Decoder가 Encoder 정보를 읽는 단계는 **Encoder–Decoder Attention(Cross-Attention)**입니다. Decoder의 Query로 Encoder 출력의 Key·Value를 참고합니다.

**Q3. Decoder 위치별로 참고 가능한 입력 위치는 무엇인가요? 위치 0에서 위치 1을 보면 어떤 정답을 미리 보게 되나요?**

- 위치 0 → {0}, 위치 1 → {0, 1}, 위치 2 → {0, 1, 2}입니다(위 표).
- 위치 1의 입력은 **00:40의 실제 기온 −8.31℃**이고, 이것은 바로 위치 0이 맞혀야 할 정답입니다. 위치 0이 위치 1을 보면 정답을 미리 보고 맞히는 셈이 됩니다.

**Q4. 현재 입력 위치를 허용해도 되는 이유와 실제 예측 시 정답이 없을 때 사용하는 값을 설명하세요. 앞 문제의 M만 계산하면 전체 Transformer 기온 예측이 완료된 것인가요?**

- Decoder 입력은 **한 칸 오른쪽으로 이동**되어 있어서, 각 위치의 현재 입력은 맞힐 정답보다 한 시점 앞의 값입니다(위치 0의 입력은 00:30 기온, 정답은 00:40 기온). 따라서 현재 위치까지 보아도 정답이 새지 않습니다.
- 실제 예측에서는 정답을 모르므로, **앞 위치에서 모델이 예측한 값**을 다음 위치의 입력으로 넣습니다(예: 00:40 예측값 → 00:50 예측의 입력).
- 완료된 것이 아닙니다. M은 Encoder 안의 Multi-Head Self-Attention 한 번에 해당하는 계산일 뿐입니다. Add & Norm, FFN, Decoder 전체, 최종 출력층, 그리고 학습이 모두 빠져 있습니다.

---

## 기초 과제. 새로운 관측 구간에 같은 구성 적용하기

### 문제 3-1. 위치 정보와 두 헤드 계산 복습하기

In [6]:
PE_home = position_encoding(len(X_home))
X_home_pos = X_home + PE_home

A1_home, H1_home = attention_head(X_home_pos, WQ1, WK1, WV1)
A2_home, H2_home = attention_head(X_home_pos, WQ2, WK2, WV2)

C_home = np.concatenate([H1_home, H2_home], axis=1)
M_home = C_home @ WO

print("PE_home:\n", np.round(PE_home, 6))
print("X_home_pos:\n", np.round(X_home_pos, 6), " shape:", X_home_pos.shape)
print("H1_home:\n", np.round(H1_home, 6), " shape:", H1_home.shape, " A1 행 합:", A1_home.sum(axis=1))
print("H2_home:\n", np.round(H2_home, 6), " shape:", H2_home.shape, " A2 행 합:", A2_home.sum(axis=1))
print("C_home:\n", np.round(C_home, 6), " shape:", C_home.shape)
print("M_home:\n", np.round(M_home, 6), " shape:", M_home.shape)
print("PE_home이 실습 PE와 같은가?:", np.array_equal(PE_home, PE))

PE_home:
 [[ 0.        1.      ]
 [ 0.841471  0.540302]
 [ 0.909297 -0.416147]]
X_home_pos:
 [[-0.831     1.942   ]
 [ 0.014471  1.481302]
 [ 0.104297  0.527853]]  shape: (3, 2)
H1_home:
 [[-0.396879]
 [-0.234847]
 [-0.219182]]  shape: (3, 1)  A1 행 합: [1. 1. 1.]
H2_home:
 [[1.752547]
 [1.692131]
 [1.485166]]  shape: (3, 1)  A2 행 합: [1. 1. 1.]
C_home:
 [[-0.396879  1.752547]
 [-0.234847  1.692131]
 [-0.219182  1.485166]]  shape: (3, 2)
M_home:
 [[0.479395 1.554108]
 [0.611218 1.574707]
 [0.523401 1.375574]]  shape: (3, 2)
PE_home이 실습 PE와 같은가?: True


#### 답변

**첫 행 계산 과정**

- 위치 덧셈: X_home[0] + PE[0] = [−0.831 + 0, 0.942 + 1] = **[−0.831, 1.942]**
- 헤드 출력: H1_home[0, 0] ≈ −0.396879, H2_home[0, 0] ≈ 1.752547
- WO 변환: [−0.396879 + 1.752547 × 0.5, −0.396879 × 0.5 + 1.752547]
  = [−0.396879 + 0.876273, −0.198439 + 1.752547] = **[0.479395, 1.554108]**

**Q1. PE_home이 실습 PE와 같은 이유는 무엇인가요? 첫 X_home_pos 행은 무엇인가요?**

- PE는 **위치 번호(0, 1, 2)만으로** 계산되고 관측값이나 시각은 사용하지 않습니다. 과제도 독립된 구간이라 위치 번호가 다시 0부터 시작하므로 PE가 같습니다(`np.array_equal` = True).
- 첫 행은 **[−0.831, 1.942]**입니다.

**Q2. 두 헤드 출력·연결 결과·최종 출력의 크기는 각각 무엇인가요? 각 Attention 가중치의 행 합은 얼마인가요?**

- H1_home·H2_home: **(3, 1)**, C_home: **(3, 2)**, M_home: **(3, 2)**입니다.
- A1_home·A2_home의 각 행 합은 모두 **1**입니다.

**Q3. 첫 위치의 두 헤드 출력과 M_home 첫 행은 무엇인가요?**

- H1 ≈ −0.396879, H2 ≈ 1.752547, M_home 첫 행 ≈ **[0.479395, 1.554108]**입니다(위 대입 과정).

**Q4. 실습과 출력 숫자가 달라진 이유는 무엇인가요? M_home을 미래 기온 예측값이라고 부를 수 있나요?**

- 위치 표현·가중치·함수는 그대로이고 **입력 관측값(00:40~01:00의 기온·습도)이 달라졌기** 때문입니다.
- 부를 수 없습니다. M_home은 이미 받은 세 관측을 위치 정보와 함께 섞은 **내부 특징 표현**입니다. 학습된 모델도 아니고 최종 출력층도 없으므로 기온 단위의 예측값이 아닙니다.

---

## 실습 마무리

**마무리 Q1. Positional Encoding이 필요한 이유, X+PE의 크기, 위치 0에서 [−0.802, 0.933]에 더하는 값을 쓰세요.**

- Self-Attention만으로는 입력의 순서를 구분하지 못하므로, 위치 정보를 표현에 더해 몇 번째 위치인지 알려 줍니다.
- X+PE의 크기는 **(3, 2)**로, 같은 위치끼리 더하므로 크기가 그대로입니다.
- 위치 0에서는 PE(0) = **[0, 1]**을 더해 [−0.802, 1.933]이 됩니다.

**마무리 Q2. 관측 수·d_model·헤드 수·각 헤드의 d_k를 쓰고, Attention 점수를 무엇으로 나누는지 설명하세요.**

- 관측 수 **3**, d_model **2**, 헤드 수 **2**, 각 헤드의 d_k **1**
- 각 헤드의 점수를 **√d_k = √1 = 1**로 나눕니다. 나누는 기준은 해당 헤드의 Key 성분 수이며, d_model이나 헤드 수가 아닙니다.

**마무리 Q3. A1·H1·C·M의 역할과 크기를 구분하세요. 행 합이 1이어야 하는 배열과 헤드 결과를 합치는 두 단계는 무엇인가요?**

| 배열 | 역할 | 크기 |
|---|---|---|
| A1 | 헤드 1의 Attention 비중 | (3, 3) |
| H1 | 헤드 1이 V를 비중으로 합친 출력 | (3, 1) |
| C | 두 헤드 출력을 옆으로 연결한 결과 | (3, 2) |
| M | C에 WO를 곱한 최종 Multi-Head 출력 | (3, 2) |

- 행 합이 1이어야 하는 것은 **A1·A2**(Softmax 비중)입니다.
- 합치는 두 단계: ① **Concat**(옆으로 연결) → ② **WO 곱하기**(출력 변환으로 다시 섞기)

**마무리 Q4. Encoder와 Decoder의 역할을 설명하고, 세 Attention의 Q·K·V 출처를 비교하세요.**

- Encoder: 입력 구간 전체의 관계를 반영한 표현을 만듭니다. Decoder: 이전 출력 정보와 Encoder 표현을 참고해 다음 출력을 차례로 만듭니다.
- Encoder Self-Attention: Q·K·V 모두 Encoder 입력
- Decoder Masked Self-Attention: Q·K·V 모두 Decoder 입력(현재 위치까지)
- Cross-Attention: Q는 Decoder, K·V는 Encoder 출력

**마무리 Q5. 00:40을 예측하는 Decoder 위치 0이 입력 위치 0은 읽어도 되고 위치 1은 읽으면 안 되는 이유와 실제 예측의 다음 입력에 쓰는 값을 적으세요.**

- 입력 위치 0은 00:30의 관측 기온(−8.51℃)으로 이미 아는 값이므로 읽어도 됩니다.
- 입력 위치 1은 00:40의 실제 기온(−8.31℃), 즉 위치 0이 맞혀야 할 정답이므로 읽으면 정답 누출입니다.
- 실제 예측에서는 정답 대신 **위치 0이 예측한 00:40 기온**을 다음 위치의 입력으로 사용합니다.

**마무리 Q6. 코드로 계산한 부분과 구조 설명으로만 다룬 부분을 나누어 쓰고, M 또는 M_home을 곧바로 미래 기온으로 해석할 수 있는지 설명하세요.**

- 코드로 계산: Positional Encoding 덧셈, 두 헤드의 Scaled Dot-Product Attention, Concat과 WO 변환(실습·과제 구간 모두)
- 구조 설명만: Add & Norm, FFN, Encoder·Decoder 전체 흐름, 마스킹, 입력 이동, 최종 출력층, 학습
- M·M_home은 고정 가중치로 계산한 **내부 특징 표현**이며 기온 단위도 아니므로, 곧바로 미래 기온으로 해석할 수 없습니다.